# Critics Index

How often does each critic review a movie during the Kalshi bet window (bet open → bet close)?
Goal: identify high-frequency critics whose reviews we can anticipate or track.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 110,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.size': 10,
})

ROOT = Path('..').resolve()

# ── Load data ─────────────────────────────────────────────────────────
mi = pd.read_csv(ROOT / 'movies_index.csv')
for col in ['Bet Open Date', 'Bet Close Date']:
    mi[col] = pd.to_datetime(mi[col], utc=True)

reviews = pd.read_csv(ROOT / 'reviews.csv')
reviews['estimated_timestamp'] = pd.to_datetime(reviews['estimated_timestamp'], utc=True, format='ISO8601')

# ── Restrict to most recent 20 movies by bet close date ──────────────
N_MOVIES = 20
recent_slugs = mi.nlargest(N_MOVIES, 'Bet Close Date')['Slug']
reviews = reviews[reviews['movie_slug'].isin(recent_slugs)].copy()

# ── Join bet close onto reviews & compute hours before close ──────────
close_map = mi.set_index('Slug')['Bet Close Date']
reviews['bet_close'] = reviews['movie_slug'].map(close_map)
reviews['hours_before_close'] = (reviews['bet_close'] - reviews['estimated_timestamp']).dt.total_seconds() / 3600

# Filter to reviews between 96h and 24h before bet close
in_window = reviews[
    (reviews['hours_before_close'] >= 24) &
    (reviews['hours_before_close'] <= 96)
].copy()

print(f"Movies (most recent {N_MOVIES}): {len(recent_slugs)}")
print(f"Reviews for those movies: {len(reviews):,}")
print(f"Reviews within 96h-24h before close: {len(in_window):,}")
print(f"Movies with in-window reviews: {in_window['movie_slug'].nunique()}")

In [ ]:
# ── Count movies reviewed per critic (within bet windows) ─────────────
critic_movie_counts = (
    in_window
    .groupby('reviewer_name')['movie_slug']
    .nunique()
    .sort_values(ascending=False)
    .rename('movies_reviewed')
)

print(f"Unique critics with in-window reviews: {len(critic_movie_counts):,}")
print(f"\nDistribution of movies-reviewed-per-critic:")
print(critic_movie_counts.describe().round(1))
print(f"\nTop 20 critics:")
critic_movie_counts.head(20)

In [ ]:
# ── Histogram: top critics by movies reviewed in bet windows ──────────
top_n = 50
top_critics = critic_movie_counts.head(top_n)

fig, ax = plt.subplots(figsize=(18, 7))
bars = ax.bar(range(len(top_critics)), top_critics.values, color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(top_critics)))
ax.set_xticklabels(top_critics.index, rotation=55, ha='right', fontsize=7)
ax.set_ylabel('Movies reviewed within window')
ax.set_xlabel('Critic')
ax.set_title(f'Top {top_n} critics by movies reviewed 96h-24h before bet close (most recent {N_MOVIES} movies)')

# Annotate bar values
for bar, val in zip(bars, top_critics.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            str(val), ha='center', va='bottom', fontsize=7)

plt.tight_layout()

In [ ]:
# ── Wider window: bet open to 24h before close, top 100 critics ───────
# Re-load unfiltered reviews for this view
reviews_all = pd.read_csv(ROOT / 'reviews.csv')
reviews_all['estimated_timestamp'] = pd.to_datetime(reviews_all['estimated_timestamp'], utc=True, format='ISO8601')

open_map = mi.set_index('Slug')['Bet Open Date']
close_map = mi.set_index('Slug')['Bet Close Date']
reviews_all['bet_open'] = reviews_all['movie_slug'].map(open_map)
reviews_all['bet_close'] = reviews_all['movie_slug'].map(close_map)
reviews_all['hours_before_close'] = (reviews_all['bet_close'] - reviews_all['estimated_timestamp']).dt.total_seconds() / 3600

# Filter: most recent 25 movies, bet open to 24h before close
reviews_wide = reviews_all[
    (reviews_all['movie_slug'].isin(recent_slugs)) &
    (reviews_all['estimated_timestamp'] >= reviews_all['bet_open']) &
    (reviews_all['hours_before_close'] >= 24)
].copy()

print(f"Reviews in window (bet open to T-24h, {N_MOVIES} movies): {len(reviews_wide):,}")
print(f"Movies with in-window reviews: {reviews_wide['movie_slug'].nunique()}")

critic_counts_wide = (
    reviews_wide
    .groupby('reviewer_name')['movie_slug']
    .nunique()
    .sort_values(ascending=False)
    .rename('movies_reviewed')
)

top_n_wide = 100
top_critics_wide = critic_counts_wide.head(top_n_wide)

fig, ax = plt.subplots(figsize=(20, 7))
bars = ax.bar(range(len(top_critics_wide)), top_critics_wide.values, color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(top_critics_wide)))
ax.set_xticklabels(top_critics_wide.index, rotation=60, ha='right', fontsize=6)
ax.set_ylabel('Movies reviewed within window')
ax.set_xlabel('Critic')
ax.set_title(f'Top {top_n_wide} critics by movies reviewed (bet open to T-24h, most recent {N_MOVIES} movies)')

for bar, val in zip(bars, top_critics_wide.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            str(val), ha='center', va='bottom', fontsize=6)

plt.tight_layout()

In [ ]:
# ── All critics (bet open to T-24h) ──────────────────────────────────
n_all = len(critic_counts_wide)
print(f"Total critics with >=1 review in window: {n_all}")

fig, ax = plt.subplots(figsize=(20, 6))
ax.bar(range(n_all), critic_counts_wide.values, color='steelblue', edgecolor='none', width=1.0)
ax.set_ylabel('Movies reviewed within window')
ax.set_xlabel(f'Critic (ranked by frequency, {n_all} total)')
ax.set_title(f'All {n_all} critics by movies reviewed (bet open to T-24h, most recent {N_MOVIES} movies)')

# Skip individual labels — too many. Show key percentiles instead.
for pct in [10, 25, 50, 75]:
    idx = int(n_all * pct / 100)
    val = critic_counts_wide.values[idx]
    ax.annotate(f'P{pct}: {val}', xy=(idx, val), xytext=(idx, val + 1),
                fontsize=8, ha='center', color='red')

plt.tight_layout()

# Summary stats
print(f"Top 10: {critic_counts_wide.values[9]}, Top 50: {critic_counts_wide.values[min(49, n_all-1)]}, "
      f"Top 100: {critic_counts_wide.values[min(99, n_all-1)]}, Top 200: {critic_counts_wide.values[min(199, n_all-1)]}")
print(f"Critics reviewing >=20/{N_MOVIES}: {(critic_counts_wide >= 20).sum()}")
print(f"Critics reviewing >=10/{N_MOVIES}: {(critic_counts_wide >= 10).sum()}")
print(f"Critics reviewing >=5/{N_MOVIES}: {(critic_counts_wide >= 5).sum()}")
print(f"Critics reviewing 1/{N_MOVIES}: {(critic_counts_wide == 1).sum()}")

In [ ]:
# ── Cumulative review share by critic rank ────────────────────────────
# How many top critics do you need to cover X% of all reviews?
total_reviews_in_window = critic_counts_wide.sum()
cumulative = critic_counts_wide.cumsum() / total_reviews_in_window

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(range(1, len(cumulative) + 1), cumulative.values, 'steelblue', lw=2)
ax.fill_between(range(1, len(cumulative) + 1), cumulative.values, alpha=0.1, color='steelblue')

# Mark key thresholds
for pct, color in [(0.50, 'orange'), (0.75, 'red'), (0.90, 'darkred')]:
    idx = int((cumulative >= pct).argmax()) + 1  # 1-indexed
    ax.axhline(pct, color=color, ls='--', alpha=0.4)
    ax.axvline(idx, color=color, ls=':', alpha=0.4)
    ax.annotate(f'{int(pct*100)}% of reviews = top {idx} critics',
                xy=(idx, pct), xytext=(idx + len(cumulative)*0.05, pct - 0.04),
                fontsize=9, color=color, arrowprops=dict(arrowstyle='->', color=color, lw=1.2))

ax.set_xlabel('Number of critics (ranked by frequency)')
ax.set_ylabel('Cumulative share of all reviews')
ax.set_title(f'Cumulative review coverage by critic rank (bet open to T-24h, most recent {N_MOVIES} movies)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.set_xlim(1, len(cumulative))
ax.set_ylim(0, 1.02)

plt.tight_layout()

print(f"Total reviews in window: {total_reviews_in_window:,}")
print(f"Total critics: {len(critic_counts_wide)}")

## Tier 1 critic timing: when do top critics review within the bet window?

Horizontal box plots showing each critic's review timing distribution (days before close).
Each dot is one review. Box shows IQR. Only reviews in the bet-open-to-T24h window.

In [ ]:
# ── Tier 1 critic timing: box + strip plot ────────────────────────────
TOP_K = 30
top_critic_names = critic_counts_wide.head(TOP_K).index.tolist()

# Get review timing for these critics (already filtered to bet open → T-24h)
tier1_reviews = reviews_wide[reviews_wide['reviewer_name'].isin(top_critic_names)].copy()
tier1_reviews['days_before_close'] = tier1_reviews['hours_before_close'] / 24

# Order critics by activity (most active at top → reversed for bottom-up y-axis)
critic_order = critic_counts_wide.head(TOP_K).index.tolist()[::-1]

fig, ax = plt.subplots(figsize=(14, max(8, TOP_K * 0.35)))

# Box plots
bp = ax.boxplot(
    [tier1_reviews[tier1_reviews['reviewer_name'] == name]['days_before_close'].values for name in critic_order],
    vert=False, positions=range(len(critic_order)), widths=0.6,
    patch_artist=True, showfliers=False,
    boxprops=dict(facecolor='steelblue', alpha=0.3, edgecolor='steelblue'),
    medianprops=dict(color='red', lw=1.5),
    whiskerprops=dict(color='steelblue', alpha=0.5),
    capprops=dict(color='steelblue', alpha=0.5),
)

# Overlay individual review dots with jitter
for i, name in enumerate(critic_order):
    vals = tier1_reviews[tier1_reviews['reviewer_name'] == name]['days_before_close'].values
    jitter = np.random.uniform(-0.15, 0.15, len(vals))
    ax.scatter(vals, i + jitter, s=12, alpha=0.5, color='steelblue', edgecolors='none', zorder=3)

ax.set_yticks(range(len(critic_order)))
ax.set_yticklabels(critic_order, fontsize=8)
ax.set_xlabel('Days before bet close')
ax.set_title(f'Top {TOP_K} critics — review timing within bet window (most recent {N_MOVIES} movies)')
ax.axvline(1, color='gray', ls=':', alpha=0.4, label='T-24h (cutoff)')
ax.set_xlim(left=0)
ax.invert_xaxis()

plt.tight_layout()

# Summary
print(f"Critics shown: {TOP_K}")
print(f"Total reviews plotted: {len(tier1_reviews)}")
median_timing = tier1_reviews.groupby('reviewer_name')['days_before_close'].median()
print(f"Median review time range: {median_timing.min():.1f} – {median_timing.max():.1f} days before close")